In [0]:
%python
from pyspark.sql import functions as F

RAW = "/Volumes/workspace/rp_landing/raw/landing"
OPS = "/Volumes/workspace/rp_landing/ops"

(spark.readStream.format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", f"{OPS}/schemas/sales")
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
  .load(f"{RAW}/sales_daily")
  .select("*", F.col("_metadata.file_path").alias("_source_file"),
          F.current_timestamp().alias("_ingest_ts"))
  .writeStream
  .option("checkpointLocation", f"{OPS}/chk/bronze_sales")
  .option("mergeSchema", "true")
  .trigger(availableNow=True)
  .toTable("workspace.rp_bronze.sales"))

In [0]:
%sql
SELECT COUNT(*) AS total_rows FROM workspace.rp_bronze.sales;

In [0]:
%sql
DESCRIBE workspace.rp_bronze.sales;

In [0]:
%sql
SELECT COUNT(*) FROM workspace.rp_bronze.sales;

In [0]:
%sql
SELECT _source_file, COUNT(*) as cnt
FROM workspace.rp_bronze.sales
GROUP BY _source_file
ORDER BY _source_file;

In [0]:
%python
files = dbutils.fs.ls("/Volumes/workspace/rp_landing/raw/landing/sales_daily")
print("Total files:", len(files))
for f in files:
    print(f.path)

In [0]:
%sql
SELECT COUNT(*) as stores_count FROM workspace.rp_bronze.stores;

In [0]:
%sql
SELECT COUNT(*) as products_count FROM workspace.rp_bronze.products;

In [0]:
%sql
SELECT 
  COUNT(*) as total,
  SUM(CASE WHEN sale_id IS NULL THEN 1 ELSE 0 END) as null_sale_id,
  SUM(CASE WHEN store_id IS NULL THEN 1 ELSE 0 END) as null_store_id,
  SUM(CASE WHEN sku IS NULL THEN 1 ELSE 0 END) as null_sku
FROM workspace.rp_bronze.sales;

In [0]:
SELECT count(*) as total_rows FROM workspace.rp_bronze.sales

In [0]:
SELECT max(sale_date) as last_day, count(*) as total_rows 
FROM workspace.rp_bronze.sales_daily